In [5]:
import pandas as pd
df = pd.read_csv("data_raw.csv")
print(df.info())
print(df.describe(exclude='object'))

nulos = df.isna().mean().sort_values(ascending=False)
dups_ratio = df.duplicated().mean()
print("nulos",nulos)
print("ratio de duplicados",dups_ratio)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   fecha      40 non-null     object 
 1   sku        40 non-null     object 
 2   canal      40 non-null     object 
 3   unidades   38 non-null     float64
 4   precio     35 non-null     float64
 5   importe    35 non-null     float64
 6   tienda_id  39 non-null     float64
dtypes: float64(4), object(3)
memory usage: 2.3+ KB
None
        unidades     precio     importe  tienda_id
count  38.000000  35.000000   35.000000  39.000000
mean    4.447368  11.914286   48.648857   1.820513
std     2.226346   3.641541   22.228785   0.823081
min     1.000000   7.250000   12.000000   1.000000
25%     3.000000  10.500000   31.750000   1.000000
50%     4.000000  10.500000   45.000000   2.000000
75%     6.000000  15.000000   60.000000   2.500000
max    10.000000  20.000000  110.000000   3.000000
nulos precio       0.1

## 2. Métricas de calidad (definición y cálculo rápido)

Una métrica de calidad convierte una sospecha en un número. No basta con decir
“hay nulos” o “hay duplicados”: hay que medir cuántos, dónde aparecen y cuánto
cambia el dataset después de tratarlos.

- **Completitud:** `1 - nulos(col)/N`
- **Unicidad (clave):** `nunique(id)/N`
- **Validez (dominio):** `mean(col.isin(conjunto))`
- **Validez (rango):** `mean(min <= col <= max)`
- **Conformidad (regex):** `mean(fullmatch(regex))`
- **Consistencia (entre columnas):** reglas lógicas (`fecha_inicio <= fecha_fin`, `importe ≈ unidades*precio`)
- **Puntualidad:** `mean(now - ts <= umbral)`

In [6]:
import numpy as np, re

completitud = 1 - df['precio'].isna().mean()
unicidad = df['sku'].nunique() / len(df)
dom_ok = df['canal'].isin(['tienda','web','app']).mean()
rng_ok = ((df['unidades']>=0)&(df['unidades']<=500)).mean()
conformidad_regex_ok = df['sku'].astype(str).str.fullmatch(r"[A-Z0-9\-]{3,16}").fillna(False).mean()
cons_ok = (np.isclose(df['importe'], df['unidades']*df['precio'], rtol=0.02, atol=0.01)).mean()

print("completitud",completitud)
print("unicidad",unicidad)
print("dom_ok",dom_ok)
print("rng_ok",rng_ok)
print("conformidad_regex_ok",conformidad_regex_ok)
print("cons_ok",cons_ok)

completitud 0.875
unicidad 0.275
dom_ok 1.0
rng_ok 0.95
conformidad_regex_ok 0.875
cons_ok 0.7


In [7]:
# Tipos y coherencias
df['fecha'] = pd.to_datetime(df['fecha'], errors='coerce')
df['unidades'] = pd.to_numeric(df['unidades'], errors='coerce').astype('Int64')
df['precio'] = pd.to_numeric(df['precio'], errors='coerce')


# Normalizar SKU abc-001 y ABC-001 deben ser el mismo SKU
df['sku'] = df['sku'].str.strip().str.upper()

# Imputación simple (ejemplo)
df['precio'] = df['precio'].fillna(df.groupby('sku')['precio'].transform('median'))


# Consistencia importe
df['importe_calc'] = df['unidades']*df['precio']
df['flag_ajuste_importe'] = ~np.isclose(df['importe'], df['importe_calc'], rtol=0.02, atol=0.01)
# |importe − importe_calc| ≤ 0.01 + 0.02 × |importe_calc|
df.loc[df['flag_ajuste_importe'], 'importe'] = df.loc[df['flag_ajuste_importe'], 'importe_calc']
# Cuidado: esto supone que el importe debe ser exactamente unidades × precio, 
# sin descuentos ni impuestos adicionales. Además, antes de comparar y sobrescribir 
# conviene separar las filas con datos ausentes.
pd.__version__

/tmp/ipykernel_71751/2605202349.py:18: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '<FloatingArray>
[43.5, 105.0, 73.5, 29.0, <NA>, 94.5, <NA>, 52.5, <NA>, <NA>, 24.0]
Length: 11, dtype: Float64' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[df['flag_ajuste_importe'], 'importe'] = df.loc[df['flag_ajuste_importe'], 'importe_calc']


'2.3.3'

In [9]:
# Pandas 
# Unificar los importes como decimales anulables de pandas
df['importe'] = pd.to_numeric(df['importe'], errors='coerce').astype('Float64')
df['importe_calc'] = (df['unidades'] * df['precio']).astype('Float64')

# Comparar únicamente filas con ambos importes disponibles
validos = df['importe'].notna() & df['importe_calc'].notna()

df['flag_ajuste_importe'] = False
df.loc[validos, 'flag_ajuste_importe'] = ~np.isclose(
    df.loc[validos, 'importe'].to_numpy(dtype='float64'),
    df.loc[validos, 'importe_calc'].to_numpy(dtype='float64'),
    rtol=0.02,
    atol=0.01,
)

# Ambos lados de la asignación tienen ahora dtype Float64
ajustar = df['flag_ajuste_importe']
df.loc[ajustar, 'importe'] = df.loc[ajustar, 'importe_calc']

#Float64 es el tipo anulable de pandas; float64 es el de NumPy. 
# La solución es convertir explícitamente,
#  no depender de conversiones implícitas durante .loc, 
# que pandas ha restringido. Documentación de pandas.

In [11]:
pd.__version__

'2.3.3'

## Key Learnings:

1. Normaliza el SKU antes de calcular estadísticas por producto.
2. Una diferencia de importe es una señal para revisar, no necesariamente un error que deba sobrescribirse.

In [10]:
completitud = 1 - df['precio'].isna().mean()
unicidad = df['sku'].nunique() / len(df)
dom_ok = df['canal'].isin(['tienda','web','app']).mean()
rng_ok = ((df['unidades']>=0)&(df['unidades']<=500)).mean()
conformidad_regex_ok = df['sku'].astype(str).str.fullmatch(r"[A-Z0-9\-]{3,16}").fillna(False).mean()
cons_ok = (np.isclose(df['importe'], df['unidades']*df['precio'], rtol=0.02, atol=0.01)).mean()

print("completitud",completitud)
print("unicidad",unicidad)
print("dom_ok",dom_ok)
print("rng_ok",rng_ok)
print("conformidad_regex_ok",conformidad_regex_ok)
print("cons_ok",cons_ok)

completitud 0.95
unicidad 0.225
dom_ok 1.0
rng_ok 1.0
conformidad_regex_ok 1.0
cons_ok 0.9


In [ ]:
import duckdb

In [12]:
import pyarrow as pa, pyarrow.parquet as pq, os

df['anio'] = df['fecha'].dt.year
df['mes'] = df['fecha'].dt.month

for (a,m), sub in df.groupby(['anio','mes']):
    path = f"data_curated/parquet/anio={a}/mes={m}"
    os.makedirs(path, exist_ok=True)
    pq.write_table(pa.Table.from_pandas(sub, preserve_index=False),
                   f"{path}/part.parquet", compression="snappy")

In [13]:
import pyarrow as pa
import pyarrow.fs as fs

print(pa.__version__)
print(pa.__file__)
print(fs.LocalFileSystem())

26.0.1
/home/jmsa/miniforge3/envs/pia-datos/lib/python3.12/site-packages/pyarrow/__init__.py


In [14]:
import sys
import pyarrow

print("Python:", sys.executable)
print("Archivo:", getattr(pyarrow, "__file__", None))
print("Rutas:", list(getattr(pyarrow, "__path__", [])))

Python: /home/jmsa/miniforge3/envs/pia-datos/bin/python
Archivo: /home/jmsa/miniforge3/envs/pia-datos/lib/python3.12/site-packages/pyarrow/__init__.py
Rutas: ['/home/jmsa/miniforge3/envs/pia-datos/lib/python3.12/site-packages/pyarrow']
